In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import json
import requests
import re

WORKFLOW = Path.cwd().parent
DATA = WORKFLOW / "data"
RESULTS = WORKFLOW / "results"

master_file = pd.read_csv(DATA / "sheet1.csv", skiprows=2, low_memory=False)

genome_loc = pd.read_csv(DATA / "sheet2.csv")

genome_loc["snp"] = ''

for index, row in genome_loc.iterrows():
    genome_loc.at[index, "snp"] = (
        f"{row['position']}_{row['reference_nucleotide']}_{row['alternative_nucleotide']}"
    )

genome_loc = genome_loc.drop(
    columns=["reference_nucleotide", "chromosome", "alternative_nucleotide", "position"]
)

genome_loc.head()

,variant,snp
0,dnaA_p.Asp3Ala,8_AT_CA
1,dnaA_p.Asp3Ala,8_A_C
2,dnaA_p.Asp3Ala,8_AT_CC
3,dnaA_p.Asp3Ala,8_AT_CG
4,dnaA_p.Asp4His,10_G_C


In [ ]:
for index, row in master_file.iterrows():
    master_file.at[index,"FINAL CONFIDENCE GRADING"] = row["FINAL CONFIDENCE GRADING"][0]

master_file = master_file.loc[
    (master_file["FINAL CONFIDENCE GRADING"] == "1")
    | (master_file["FINAL CONFIDENCE GRADING"] == "2")
]

master_file = master_file[
    [
        "variant",
        "drug",
        "FINAL CONFIDENCE GRADING"
    ]
]

master_file.head()

,variant,drug,FINAL CONFIDENCE GRADING
430,eis_c.-14C>T,Amikacin,2
888,rrs_n.1401A>G,Amikacin,1
890,rrs_n.1402C>T,Amikacin,2
940,rrs_n.1484G>T,Amikacin,1
2198,atpE_p.Ala63Pro,Bedaquiline,2


In [ ]:
master_file = pd.merge(master_file, genome_loc, on="variant", how='inner')
master_file.head()

In [ ]:
'''
weight_dict = {i:{snp:0 for snp in master_file['snp'].unique()} for i in master_file['drug'].unique()}

for key, value in weight_dict.items():
    subset = master_file[master_file['drug'] == key]
    for index, row in subset.iterrows():
        weight_dict[row['drug']][row['snp']] = 1 
'''
        
weight_dict = {}

for _, row in master_file.iterrows():
    drug = row["drug"]
    snp = row["snp"]

    weight_dict.setdefault(drug, {})[snp] = 1

weight_dict

{'Amikacin': {'2715344_GTG_ATA': 1,
  '2715345_TG_CA': 1,
  '2715346_G_A': 1,
  '2715346_GGC_AGG': 1,
  '1473246_A_G': 1,
  '1473247_C_T': 1,
  '1473329_G_T': 1},
 'Bedaquiline': {'1461231_GCA_CCG': 1,
  '1461231_G_C': 1,
  '1461231_GCA_CCC': 1,
  '1461231_GCA_CCT': 1,
  '1461127_AC_CT': 1,
  '1461127_AC_CA': 1,
  '1461127_AC_CG': 1,
  '1461127_A_C': 1,
  '1461125_TGA_GGG': 1,
  '1461127_A_G': 1,
  '1461127_AC_GT': 1,
  '1461127_AC_GA': 1,
  '1461127_AC_GG': 1,
  '1461127_AC_TA': 1,
  '1461127_AC_TT': 1,
  '1461127_AC_TG': 1,
  '1461127_A_T': 1,
  '1461227_G_C': 1,
  '1461227_G_T': 1,
  '1461242_C_G': 1,
  '2859292_TGGGTCTCCTACAGAATGGCCAGTTCTTTGGGGAACCGGGTCAACAATTCCGGGGTCTGCCCGGCGGTTTCAGGCATTTTCGGCGTCCCGCCAGCCACTACCAATGTGTCCTCGATGCGGACACCGCCGCGGCCGGGTAAATAGACACCGGGCTCCACGGTCACCACGGAGCCCGCCAGTAGTGTACCGGCGGATGTGACCCCGATGCCCGGCGCTTCATGTATCTGCAGGCCAACACCGTGTCCCAGTCCGTGACCGAAGTGCTCGCCGTAGCCGGCGTCGGCGATCAGCTGGCGCGCTGCAGCGTCCACCCCCCGCAGCTCGGCACCCGGCAGCAACGCCTGCCGACCGGCCTGTTGCGCCTCGGCCACCAGCTG

In [ ]:
drug_dict = {}


for _, row in master_file.iterrows():
    snp = row["snp"]
    drug = row["drug"]
    grade = row["FINAL CONFIDENCE GRADING"]
    
    # Initialize nested dict if needed
    if snp not in drug_dict:
        drug_dict[snp] = {}
    
    # Initialize list if needed
    if drug not in drug_dict[snp]:
        drug_dict[snp][drug] = []
    
    # Append the grade
    drug_dict[snp][drug].append(grade)


drug_dict

{'2715344_GTG_ATA': {'Amikacin': ['2'], 'Kanamycin': ['1', '1']},
 '2715345_TG_CA': {'Amikacin': ['2'], 'Kanamycin': ['1']},
 '2715346_G_A': {'Amikacin': ['2'], 'Kanamycin': ['1']},
 '2715346_GGC_AGG': {'Amikacin': ['2'], 'Kanamycin': ['1']},
 '1473246_A_G': {'Amikacin': ['1'], 'Capreomycin': ['1'], 'Kanamycin': ['1']},
 '1473247_C_T': {'Amikacin': ['2'], 'Capreomycin': ['1'], 'Kanamycin': ['2']},
 '1473329_G_T': {'Amikacin': ['1'], 'Capreomycin': ['1'], 'Kanamycin': ['1']},
 '1461231_GCA_CCG': {'Bedaquiline': ['2']},
 '1461231_G_C': {'Bedaquiline': ['2']},
 '1461231_GCA_CCC': {'Bedaquiline': ['2']},
 '1461231_GCA_CCT': {'Bedaquiline': ['2']},
 '1461127_AC_CT': {'Bedaquiline': ['2']},
 '1461127_AC_CA': {'Bedaquiline': ['2']},
 '1461127_AC_CG': {'Bedaquiline': ['2']},
 '1461127_A_C': {'Bedaquiline': ['2']},
 '1461125_TGA_GGG': {'Bedaquiline': ['2']},
 '1461127_A_G': {'Bedaquiline': ['2']},
 '1461127_AC_GT': {'Bedaquiline': ['2']},
 '1461127_AC_GA': {'Bedaquiline': ['2']},
 '1461127_AC_G

In [ ]:
final_dict = {"NC_000962.3": drug_dict}

json_str = json.dumps(final_dict)

with open(RESULTS/"snp-grading.json", "w") as f:
    f.write(json_str)

In [ ]:
weight_str = json.dumps(weight_dict)

with open(RESULTS/"weight_matrix.json", "w") as file:
    file.write(weight_str)

In [ ]:
for val in drug_dict.values():
    for grad in val.values():
        if len(grad) > 1:
            print(grad)

['1', '1']
['1', '2']
['1', '2']
['1', '2']
['1', '2']
['1', '1']
['1', '1']
['1', '1']
['2', '2']
['2', '1']
['2', '2', '2', '2']
['2', '2']
['2', '2']
['2', '2']
['1', '2']
['1', '2']
['1', '2']
['1', '2']
['2', '2']
['2', '2']
['2', '2']
['1', '2']
['1', '2']
['2', '2']
['1', '2']
['1', '2']
['2', '2']
